# 04: Backtest and Walk-Forward Evaluation

This notebook demonstrates the walk-forward backtesting framework for evaluating portfolio strategies on out-of-sample data. Walk-forward testing is the standard methodology for avoiding look-ahead bias and assessing realistic strategy performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import portfolio optimization modules
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from portfolio_opt.data import (
    build_default_price_data,
    clean_price_frame,
    estimate_expected_returns,
    estimate_covariance,
)
from portfolio_opt.portfolio import (
    discrete_objective,
    solve_mvo,
    exact_enumeration,
)
from portfolio_opt.ga import genetic_algorithm
from portfolio_opt.sa import simulated_annealing
from portfolio_opt.qa import run_qaoa_on_qubo
from portfolio_opt.backtest import train_test_split, walk_forward_backtest
from portfolio_opt.metrics import sharpe_ratio, max_drawdown, portfolio_return, portfolio_volatility

print("All modules loaded successfully.")

## Data Preparation

In [ ]:
# Build deterministic price data with longer history for walk-forward testing
prices = build_default_price_data(n_assets=4, n_days=240, seed=42)
prices_clean, clean_report = clean_price_frame(prices)
returns = prices_clean.pct_change().dropna()

print(f"Total returns shape: {returns.shape}")
print(f"Observation period: {returns.index[0]} to {returns.index[-1]}")
print(f"\nData cleaning report: {clean_report}")

# Show return statistics
print(f"\nReturn statistics:")
print(returns.describe())

## Walk-Forward Test Configuration

In [ ]:
# Configuration for walk-forward testing
train_window = 120  # 120 trading days (~6 months)
test_window = 30    # 30 trading days (~1 month)
k = 2  # Select 2 assets
lambda_risk = 1.0

print(f"Walk-Forward Configuration:")
print(f"  Training window: {train_window} days")
print(f"  Test (rebalance) window: {test_window} days")
print(f"  Assets to select: {k}")
print(f"  Risk aversion: {lambda_risk}")
print(f"  Total observation windows: {len(returns) - train_window - test_window + 1}")

## Define Strategy Functions

Each strategy takes (train_returns, test_returns) and returns performance metrics.

In [ ]:
def strategy_exact(train_returns, test_returns):
    """Exact enumeration strategy: optimize on training data, evaluate on test."""
    mu = estimate_expected_returns(train_returns)
    cov = estimate_covariance(train_returns)
    x, obj_train = exact_enumeration(mu, cov, k, risk_aversion=lambda_risk)
    
    # Evaluate on test returns
    test_weights = x / x.sum() if x.sum() > 0 else x
    portfolio_rets = (test_returns @ test_weights)
    
    return {
        'strategy': 'Exact',
        'train_objective': obj_train,
        'test_return': portfolio_rets.mean() * 252,  # Annualize
        'test_volatility': portfolio_rets.std() * np.sqrt(252),
        'test_sharpe': sharpe_ratio(portfolio_rets.values, annual=True) if portfolio_rets.std() > 0 else 0,
        'selected': list(np.where(x)[0]),
    }

def strategy_mvo(train_returns, test_returns):
    """MVO continuous strategy."""
    mu = estimate_expected_returns(train_returns)
    cov = estimate_covariance(train_returns)
    w = solve_mvo(mu, cov, k=None)
    x = np.zeros(len(mu))
    x[np.argsort(w)[-k:]] = 1
    obj_train = discrete_objective(x, mu, cov, k, risk_aversion=lambda_risk)
    
    test_weights = x / x.sum() if x.sum() > 0 else x
    portfolio_rets = (test_returns @ test_weights)
    
    return {
        'strategy': 'MVO',
        'train_objective': obj_train,
        'test_return': portfolio_rets.mean() * 252,
        'test_volatility': portfolio_rets.std() * np.sqrt(252),
        'test_sharpe': sharpe_ratio(portfolio_rets.values, annual=True) if portfolio_rets.std() > 0 else 0,
        'selected': list(np.where(x)[0]),
    }

def strategy_ga(train_returns, test_returns):
    """Genetic algorithm strategy."""
    mu = estimate_expected_returns(train_returns)
    cov = estimate_covariance(train_returns)
    result = genetic_algorithm(mu, cov, k, population_size=20, generations=30, seed=42, risk_aversion=lambda_risk)
    x = result['x']
    
    test_weights = x / x.sum() if x.sum() > 0 else x
    portfolio_rets = (test_returns @ test_weights)
    
    return {
        'strategy': 'GA',
        'train_objective': result['objective'],
        'test_return': portfolio_rets.mean() * 252,
        'test_volatility': portfolio_rets.std() * np.sqrt(252),
        'test_sharpe': sharpe_ratio(portfolio_rets.values, annual=True) if portfolio_rets.std() > 0 else 0,
        'selected': list(np.where(x)[0]),
    }

def strategy_sa(train_returns, test_returns):
    """Simulated annealing strategy."""
    mu = estimate_expected_returns(train_returns)
    cov = estimate_covariance(train_returns)
    result = simulated_annealing(mu, cov, k, iterations=300, seed=42, risk_aversion=lambda_risk)
    x = result['x']
    
    test_weights = x / x.sum() if x.sum() > 0 else x
    portfolio_rets = (test_returns @ test_weights)
    
    return {
        'strategy': 'SA',
        'train_objective': result['objective'],
        'test_return': portfolio_rets.mean() * 252,
        'test_volatility': portfolio_rets.std() * np.sqrt(252),
        'test_sharpe': sharpe_ratio(portfolio_rets.values, annual=True) if portfolio_rets.std() > 0 else 0,
        'selected': list(np.where(x)[0]),
    }

def strategy_qaoa(train_returns, test_returns):
    """QAOA strategy."""
    mu = estimate_expected_returns(train_returns)
    cov = estimate_covariance(train_returns)
    result = run_qaoa_on_qubo(mu, cov, k, depth=1, shots=256, seed=42, risk_aversion=lambda_risk)
    x = result['x']
    
    test_weights = x / x.sum() if x.sum() > 0 else x
    portfolio_rets = (test_returns @ test_weights)
    
    return {
        'strategy': 'QAOA',
        'train_objective': result['objective'],
        'test_return': portfolio_rets.mean() * 252,
        'test_volatility': portfolio_rets.std() * np.sqrt(252),
        'test_sharpe': sharpe_ratio(portfolio_rets.values, annual=True) if portfolio_rets.std() > 0 else 0,
        'selected': list(np.where(x)[0]),
    }

print("Strategy functions defined.")

## Run Walk-Forward Backtest

In [ ]:
# Run walk-forward evaluation for each strategy
strategies = [
    ('Exact', strategy_exact),
    ('MVO', strategy_mvo),
    ('GA', strategy_ga),
    ('SA', strategy_sa),
    ('QAOA', strategy_qaoa),
]

wf_results = {}
for name, strategy_fn in strategies:
    print(f"\nRunning walk-forward backtest for {name}...")
    results = walk_forward_backtest(returns, strategy_fn, train_window=train_window, test_window=test_window)
    wf_results[name] = results
    print(f"  Completed {len(results)} windows")
    print(f"  Avg test Sharpe: {np.mean([r['test_sharpe'] for r in results]):.4f}")

print("\nWalk-forward backtesting complete.")

## Aggregate Performance Summary

In [ ]:
# Aggregate results across all walk-forward windows
summary_rows = []
for strategy_name, results in wf_results.items():
    test_returns = [r['test_return'] for r in results]
    test_vols = [r['test_volatility'] for r in results]
    test_sharpes = [r['test_sharpe'] for r in results]
    train_objs = [r['train_objective'] for r in results]
    
    summary_rows.append({
        'Strategy': strategy_name,
        'Avg Train Obj': np.mean(train_objs),
        'Avg Test Return (Ann.)': np.mean(test_returns),
        'Avg Test Volatility (Ann.)': np.mean(test_vols),
        'Avg Test Sharpe': np.mean(test_sharpes),
        'Std Dev Sharpe': np.std(test_sharpes),
        'Windows': len(results),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*100)
print("WALK-FORWARD BACKTEST SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

## Visualization: Out-of-Sample Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sharpe ratio across windows
ax = axes[0, 0]
for strategy_name, results in wf_results.items():
    sharpes = [r['test_sharpe'] for r in results]
    ax.plot(sharpes, marker='o', label=strategy_name, linewidth=2)
ax.set_xlabel('Rebalance Window', fontsize=11, fontweight='bold')
ax.set_ylabel('Sharpe Ratio', fontsize=11, fontweight='bold')
ax.set_title('Out-of-Sample Sharpe Ratio Across Windows', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, linestyle='--')

# Return across windows
ax = axes[0, 1]
for strategy_name, results in wf_results.items():
    returns_list = [r['test_return'] for r in results]
    ax.plot(returns_list, marker='s', label=strategy_name, linewidth=2)
ax.set_xlabel('Rebalance Window', fontsize=11, fontweight='bold')
ax.set_ylabel('Annualized Return', fontsize=11, fontweight='bold')
ax.set_title('Out-of-Sample Annualized Returns', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, linestyle='--')

# Box plot: Sharpe distribution
ax = axes[1, 0]
sharpe_data = [wf_results[s][0:10] if len(wf_results[s]) > 0 else [] for s in wf_results.keys()]
sharpe_lists = [[r['test_sharpe'] for r in wf_results[name]] for name in wf_results.keys()]
ax.boxplot(sharpe_lists, labels=list(wf_results.keys()))
ax.set_ylabel('Sharpe Ratio', fontsize=11, fontweight='bold')
ax.set_title('Distribution of Out-of-Sample Sharpe Ratios', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Average metrics comparison
ax = axes[1, 1]
strat_names = list(summary_df['Strategy'])
avg_sharpes = list(summary_df['Avg Test Sharpe'])
colors_map = {'Exact': 'green', 'MVO': 'blue', 'GA': 'orange', 'SA': 'purple', 'QAOA': 'red'}
colors = [colors_map.get(s, 'gray') for s in strat_names]
bars = ax.bar(strat_names, avg_sharpes, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Sharpe Ratio', fontsize=11, fontweight='bold')
ax.set_title('Average Out-of-Sample Sharpe Ratio', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
for bar, val in zip(bars, avg_sharpes):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print("\nVisualizations created.")

## Key Observations

1. **Walk-Forward Design**: Each window trains on the prior 120 days and tests on the next 30 days, avoiding look-ahead bias.

2. **Out-of-Sample Validation**: Unlike in-sample metrics computed during training, these metrics reflect realistic future performance.

3. **Strategy Stability**: Strategies showing high variance in Sharpe ratios across windows may be overfitting to specific periods.

4. **Return vs Volatility Trade-off**: Higher returns typically come with higher volatility. The Sharpe ratio normalizes for this trade-off.

5. **Quantum vs Classical**: QAOA performance reflects both the quality of the sampled bitstrings and the feasibility of cardinality constraints across rebalancing periods.

### Practical Takeaway
Walk-forward testing is the gold standard for evaluating portfolio strategies because:
- It prevents look-ahead bias by using only past data for optimization
- It simulates realistic rebalancing decisions
- It reveals whether strategies generalize to unseen market regimes